In [0]:
%run "../../utils/setup_context"

In [0]:
def execute_func():Unit={
  import org.apache.spark.sql.SaveMode

  val df =
    spark.sql(
      """
        |WITH community AS (
        |  SELECT tag_id FROM revr_bmbs_dmp_offline.nonmmcstore_dailytags
        |),
        |
        |comb AS (
        |  SELECT
        |    unifyid.id   AS unify_id,
        |    unifyid.type AS id_type,
        |    tl.col.tagId AS tag_id,
        |    tl.col.score AS tag_score,
        |    dt           AS lst_mdf_dt
        |  FROM revr_bmbs_dmp_offline.rv_dmp_offline_tags_comb
        |  LATERAL VIEW explode(tagRecord.tags) tl
        |  WHERE dt = date_format(date_add(current_date(), -1), 'yyyyMMdd')
        |    AND unifyid.type < 2
        |),
        |
        |daily AS (
        |  SELECT
        |    unifyid.id   AS unify_id,
        |    unifyid.type AS id_type,
        |    tl.col.tagId AS tag_id,
        |    tl.col.score AS tag_score,
        |    dt           AS lst_mdf_dt
        |  FROM revr_bmbs_dmp_offline.rv_dmp_offline_tags_daily
        |  LATERAL VIEW explode(tagRecord.tags) tl
        |  WHERE dt = date_format(date_add(current_date(), -1), 'yyyyMMdd')
        |    AND unifyid.type < 2
        |)
        |
        |SELECT c.*
        |FROM comb c
        |LEFT JOIN revr_bmbs_dmp_offline.dmp_tag t
        |  ON c.tag_id = t.tag_id
        |WHERE t.tag_name NOT LIKE '%MMC%'
        |  AND c.tag_id NOT IN (SELECT tag_id FROM community)
        |
        |UNION ALL
        |
        |SELECT d.*
        |FROM daily d
        |LEFT JOIN revr_bmbs_dmp_offline.dmp_tag t
        |  ON d.tag_id = t.tag_id
        |WHERE t.tag_name LIKE '%MMC%'
        |
        |UNION ALL
        |
        |SELECT d.*
        |FROM daily d
        |WHERE d.tag_id IN (SELECT tag_id FROM community)
        |""".stripMargin).write.mode(SaveMode.Overwrite).insertInto("revr_bmbs_dmp_offline.dmp_offline_tagengine_exploded")

  spark.sql(
      """
        |select
        |    cdm_cust_vhcl_realationship.customer_ucid ucid,
        |    vhcl_srvc.chassis as fin,
        |    vhcl_srvc.vin,
        |    vhcl_srvc.gs_code_purchase,
        |    vhcl_srvc.gs_code_lst
        |from revr_bmbs_dmp_offline.rcrd_vhcl_srvc vhcl_srvc
        |left join
        |    revr_bmbs_dmp_offline.dw_rltnshp_cdm_cust_vhcl_relationship_latest_record cdm_cust_vhcl_realationship
        |        on cdm_cust_vhcl_realationship.fin=vhcl_srvc.chassis
        |""".stripMargin).createOrReplaceTempView("temp_relationship_view1")

  spark.sql(
      """
        |select
        |    trv1.*,
        |    concat(cpd_customerinfo.last_name1,cpd_customerinfo.first_name) as customer_name,
        |    cpd_customerinfo.mobile_phone as mobilephone,
        |    cpd_customerinfo.ciam_id as ciam_id
        |from
        |    temp_relationship_view1 trv1
        |left join
        |    revr_bmbs_dmp_offline.dw_cust_cpd_customerinfo_latest_record cpd_customerinfo
        |        on trv1.vin=cpd_customerinfo.vin
        |""".stripMargin).createOrReplaceTempView("temp_relationship_view2")

  spark.sql(
      """
        |select
        |    ucid,
        |    fin,
        |    case
        |        when vin is not null
        |        and vin not in ('null' ,
        |        'NULL',
        |        '') then vin
        |        else ''
        |    end vin,
        |    case
        |        when customer_name is not null
        |        and customer_name not in ('null' ,
        |        'NULL',
        |        '') then customer_name
        |        else ''
        |    end customer_name,
        |    case
        |        when mobilephone is not null
        |        and mobilephone not in ('null' ,
        |        'NULL',
        |        '') then mobilephone
        |        else ''
        |    end mobilephone,
        |    case
        |        when ciam_id is not null
        |        and ciam_id not in ('null' ,
        |        'NULL',
        |        '') then ciam_id
        |        else ''
        |    end ciam_id,
        |    gs_code_purchase             as purchase_dealer,
        |    gs_code_lst                  as last_service_dealer
        |from
        |    temp_relationship_view2
        |""".stripMargin).write.mode(SaveMode.Overwrite).insertInto("revr_bmbs_dmp_offline.rcrd_cust_vhcl_info")

  spark.sql(
      """
        |select
        |ucid,
        |d_p.dealer_gs_code,
        |row_number()over(partition by ucid order by test_drive_time desc) as rn
        |from (
        |    select
        |		  distinct
        |		  lead.id
        |		  ,t3.ucid
        |    from datalake_otrplus.dw_cust_otrplus_lead_lead lead
        |    left join (
        |        select distinct
        |			    id
        |			    ,ucid
        |        from datalake_otrplus.dw_cust_otrplus_customer
        |    )t3
        |    on lead.customer_id=t3.id
        |) lead
        |left join (
        |	  select
        |	  	lead_id
        |	  	,dealer_id
        |	  	,from_unixtime(updated_time/1000,'yyyy-MM-dd') as test_drive_time
        |	  from datalake_otrplus.dw_cust_otrplus_lead_test_drive
        |) as test_drive
        |on lead.id = test_drive.lead_id
        |left join (
        |	select
        |		distinct
        |    dealer_id
        |    ,dealer_gs_code
        |  from revr_bmbs_dmp_offline.dealer_id_gs_dms_mapping
        |) d_p
        |on test_drive.dealer_id=d_p.dealer_id
        |""".stripMargin).createOrReplaceTempView("temp_test_drive_dealer_view")

  spark.sql(
      """
        |   select
        |   ucid
        |   ,d_p.dealer_gs_code
        |   ,row_number()over(partition by ucid order by visit_time desc) as rn
        |   from (
        |       select
        |   		distinct
        |   		lead.id
        |   		,t3.ucid
        |       from datalake_otrplus.dw_cust_otrplus_lead_lead lead
        |       left join (
        |           select
        |   			distinct
        |   			id
        |   			,ucid
        |           from datalake_otrplus.dw_cust_otrplus_customer
        |       )t3
        |       on lead.customer_id=t3.id
        |   ) lead
        |   left join (
        |       select
        |           lead_id
        |   		,dealer_id
        |   		,from_unixtime(arrival_time/1000,'yyyy-MM-dd') as visit_time
        |       from datalake_otrplus.dw_cust_otrplus_lead_showroom_visit
        |   ) as showroom_visit
        |   on lead.id = showroom_visit.lead_id
        |   left join (
        |   	SELECT
        |   		distinct dealer_id,dealer_gs_code
        |       FROM revr_bmbs_dmp_offline.dealer_id_gs_dms_mapping
        |   ) d_p
        |   on showroom_visit.dealer_id=d_p.dealer_id
        |""".stripMargin).createOrReplaceTempView("temp_showroom_visit_dealer_view")

  spark.sql(
      """
        |select
        |cdm.ucid,
        |cdm.mobilephone,
        |cdm.lastname,
        |cdm.firstname,
        |substring(trim(cvr.globalvin),1,17) as fin,
        |substring(trim(cvr.localvin),1,17) as vin,
        |cdm_digital.ciamid as ciam_id,
        |vhcl_srvc.gs_code_purchase,
        |vhcl_srvc.gs_code_lst
        |from (
        |  select
        |  ucid
        |  ,mobilephone
        |  ,firstname
        |  ,lastname
        |  from datalake_cdm.dw_cust_cdm_accnt
        |) cdm
        |left join
        |    revr_bmbs_dmp_offline.dw_rltnshp_cdm_cust_vhcl_relationship_latest_record cvr
        |        on cvr.customer_ucid=cdm.ucid
        |left join
        |    revr_bmbs_dmp_offline.rcrd_vhcl_srvc vhcl_srvc
        |        on substring(trim(cvr.globalvin),1,17)=vhcl_srvc.chassis
        |left join (
        |  select
        |   onliemobile
        |   ,ciamid
        |   ,row_number()over(PARTITION BY onliemobile order by ciamid desc) as rn
        |  from datalake_cdm.dw_cust_cdm_digital_accnt
        |) cdm_digital
        |on cdm.mobilephone=cdm_digital.onliemobile and cdm_digital.rn=1
        |""".stripMargin).createOrReplaceTempView("temp_cdm_relationship_view")

  spark.sql(
      """
        |select
        |    t_r_v.ucid,
        |    fin,
        |    case
        |        when vin is not null
        |        and vin not in ('null' ,
        |        'NULL',
        |        '') then vin
        |        else ''
        |    end vin,
        |    case
        |        when firstname is not null
        |        and firstname not in ('null' ,
        |        'NULL',
        |        '') then firstname
        |        else ''
        |    end firstname,
        |    case
        |        when lastname is not null
        |        and lastname not in ('null' ,
        |        'NULL',
        |        '') then lastname
        |        else ''
        |    end lastname,
        |    case
        |        when mobilephone is not null
        |        and mobilephone not in ('null' ,
        |        'NULL',
        |        '') then mobilephone
        |        else ''
        |    end mobilephone,
        |    case
        |        when ciam_id is not null
        |        and ciam_id not in ('null' ,
        |        'NULL',
        |        '') then ciam_id
        |        else ''
        |    end ciam_id,
        |    t_r_v.gs_code_purchase             as purchase_dealer,
        |    t_r_v.gs_code_lst                  as last_service_dealer,
        |    last_test_drive.dealer_gs_code     as last_testdrive_dealer,
        |    last_showroom_visit.dealer_gs_code as last_showroom_visit_dealer
        |from
        |    temp_cdm_relationship_view t_r_v
        |left join
        |    temp_test_drive_dealer_view last_test_drive
        |       on last_test_drive.ucid=t_r_v.ucid and last_test_drive.rn=1
        |left join
        |    temp_showroom_visit_dealer_view last_showroom_visit
        |       on last_showroom_visit.ucid=t_r_v.ucid and last_showroom_visit.rn=1
        |""".stripMargin).write.mode(SaveMode.Overwrite).insertInto("revr_bmbs_dmp_offline.dcff_cust_cdm_info")

  spark.sql(
      """
        |select
        |    tag_exploded.tag_id,
        |    tag_exploded.unify_id,
        |    tag_exploded.id_type,
        |    tag_exploded.tag_score,
        |    cvi.customer_name last_name,
        |    '' first_name,
        |    cvi.mobilephone mobile_phone,
        |    cvi.vin,
        |    cvi.fin,
        |    cvi.ciam_id
        |from
        |    (select
        |        *
        |    from
        |        revr_bmbs_dmp_offline.dmp_offline_tagengine_exploded
        |    where
        |        id_type=0) tag_exploded
        |left join
        |    revr_bmbs_dmp_offline.rcrd_cust_vhcl_info cvi
        |        on tag_exploded.unify_id=cvi.ucid
        |union
        |select
        |    tag_exploded.tag_id,
        |    tag_exploded.unify_id,
        |    tag_exploded.id_type,
        |    tag_exploded.tag_score,
        |    cvi.customer_name last_name,
        |    '' first_name,
        |    cvi.mobilephone mobile_phone,
        |    cvi.vin,
        |    cvi.fin,
        |    cvi.ciam_id
        |from
        |    (select
        |        *
        |    from
        |        revr_bmbs_dmp_offline.dmp_offline_tagengine_exploded
        |    where
        |        id_type=1) tag_exploded
        |left join
        |    revr_bmbs_dmp_offline.rcrd_cust_vhcl_info cvi
        |        on tag_exploded.unify_id=cvi.fin
        |where
        |!((nullif(cvi.vin,'') is null)
        |and (nullif(cvi.fin,'') is null))
        |""".stripMargin).write.mode(SaveMode.Overwrite).insertInto("revr_bmbs_dmp_offline.dmp_offline_tags_with_info")

/*  spark.sql(
      """
        |select *,
        |date_format(date_add(current_date,-1),'yyyyMMdd') as dt
        |from revr_bmbs_dmp_offline.dmp_offline_tags_with_info
        |""".stripMargin).write.mode(SaveMode.Overwrite).insertInto("revr_bmbs_dmp_offline.dmp_offline_tags_with_info_partition")
  send_slack_info_base_on_query_with_spark("select dt,count(1) from revr_bmbs_dmp_offline.dmp_offline_tags_with_info_partition where dt=date_format(date_add(current_date,-1),'yyyyMMdd') group by dt","revr_bmbs_dmp_offline.dmp_offline_tags_with_info_partition record in dt(%s) count:%s")
*/
  // add a new view to

  spark.sql(
    """
      |select ucid,lastname,firstname, mobilephone,salutation,case when id_gender is null then gender else id_gender end gender from (select ucid,lastname,firstname, mobilephone,salutation,case
      |        when length(id_num)              =18
      |            and substring(id_num,17,1)%2 = 1
      |            then 'M'
      |        when length(id_num)              =18
      |            and substring(id_num,17,1)%2 = 0
      |            then 'F'
      |        when length(id_num)              =15
      |            and substring(id_num,15,1)%2 = 1
      |            then 'M'
      |        when length(id_num)              =15
      |            and substring(id_num,15,1)%2 = 0
      |            then 'F'
      |        end as id_gender,gender from (
      |    select ucid, lastname,firstname, mobilephone,salutation, if(identifier1_type = 'ID Card/Driving Card'
      |            and
      |            (
      |                length(identifier1_number)    = 18
      |                or length(identifier1_number) = 15
      |            )
      |            ,identifier1_number,null) as id_num,gender from datalake_cdm.dw_cust_cdm_accnt) )
      |""".stripMargin).createOrReplaceTempView("temp_cust_cdm_view")

  spark.sql(
      """
        |select src.tag_id,
        |       src.unify_id,
        |       src.id_type,
        |       src.tag_score,
        |       src.ucid,
        |       src.last_name,
        |       src.first_name,
        |       src.mobile_phone,
        |       src.vin,
        |       src.fin,
        |       src.ciam_id,
        |       src.purchase_dealer,
        |       src.last_service_dealer,
        |       src.last_testdrive_dealer,
        |       src.last_showroom_visit_dealer,
        |       case when cust.salutation not in ('null','NULL','') then cust.salutation  else null end salutation,
        |       case when cust.gender not in ('null','NULL','') then cust.gender  else null end gender
        |   from (select
        |    tag_exploded.tag_id,
        |    tag_exploded.unify_id,
        |    tag_exploded.id_type,
        |    tag_exploded.tag_score,
        |    case when cvi.ucid not in ('null','NULL','')                       then cvi.ucid                       else null end as ucid,
        |    case when cvi.lastname not in ('null','NULL','')                   then cvi.lastname                   else null end as last_name,
        |    case when cvi.firstname not in ('null','NULL','')                  then cvi.firstname                  else null end as first_name,
        |    case when cvi.mobilephone not in ('null','NULL','')                then cvi.mobilephone                else null end as mobile_phone,
        |    case when cvi.vin not in ('null','NULL','')                        then cvi.vin                        else null end as vin,
        |    case when cvi.fin not in ('null','NULL','')                        then cvi.fin                        else null end as fin,
        |    case when cvi.ciam_id not in ('null','NULL','')                    then cvi.ciam_id                    else null end as ciam_id,
        |    case when cvi.purchase_dealer not in ('null','NULL','')            then cvi.purchase_dealer            else null end as purchase_dealer,
        |    case when cvi.last_service_dealer not in ('null','NULL','')        then cvi.last_service_dealer        else null end as last_service_dealer,
        |    case when cvi.last_testdrive_dealer not in ('null','NULL','')      then cvi.last_testdrive_dealer      else null end as last_testdrive_dealer,
        |    case when cvi.last_showroom_visit_dealer not in ('null','NULL','') then cvi.last_showroom_visit_dealer else null end as last_showroom_visit_dealer
        |from
        |    (select
        |        *
        |    from
        |        revr_bmbs_dmp_offline.dmp_offline_tagengine_exploded
        |    where
        |        id_type=0) tag_exploded
        |left join
        |    revr_bmbs_dmp_offline.dcff_cust_cdm_info cvi
        |        on tag_exploded.unify_id=cvi.ucid
        |union
        |select
        |    tag_exploded.tag_id,
        |    tag_exploded.unify_id,
        |    tag_exploded.id_type,
        |    tag_exploded.tag_score,
        |    case when cvi.ucid not in ('null','NULL','')                       then cvi.ucid                       else null end as ucid,
        |    case when cvi.lastname not in ('null','NULL','')                   then cvi.lastname                   else null end as last_name,
        |    case when cvi.firstname not in ('null','NULL','')                  then cvi.firstname                  else null end as first_name,
        |    case when cvi.mobilephone not in ('null','NULL','')                then cvi.mobilephone                else null end as mobile_phone,
        |    case when cvi.vin not in ('null','NULL','')                        then cvi.vin                        else null end as vin,
        |    case when cvi.fin not in ('null','NULL','')                        then cvi.fin                        else null end as fin,
        |    case when cvi.ciam_id not in ('null','NULL','')                    then cvi.ciam_id                    else null end as ciam_id,
        |    case when cvi.purchase_dealer not in ('null','NULL','')            then cvi.purchase_dealer            else null end as purchase_dealer,
        |    case when cvi.last_service_dealer not in ('null','NULL','')        then cvi.last_service_dealer        else null end as last_service_dealer,
        |    case when cvi.last_testdrive_dealer not in ('null','NULL','')      then cvi.last_testdrive_dealer      else null end as last_testdrive_dealer,
        |    case when cvi.last_showroom_visit_dealer not in ('null','NULL','') then cvi.last_showroom_visit_dealer else null end as last_showroom_visit_dealer
        |from
        |    (select
        |        *
        |    from
        |        revr_bmbs_dmp_offline.dmp_offline_tagengine_exploded
        |    where
        |        id_type=1) tag_exploded
        |left join (
        |  select
        |     rcv.ucid,
        |     rcv.fin,
        |     rcv.vin,
        |     cdm.lastname,
        |     cdm.firstname,
        |     cdm.mobilephone,
        |     cdm_digital.ciamid as ciam_id,
        |     rcv.purchase_dealer,
        |     rcv.last_service_dealer,
        |     last_test_drive.dealer_gs_code     as last_testdrive_dealer,
        |     last_showroom_visit.dealer_gs_code as last_showroom_visit_dealer
        |  from revr_bmbs_dmp_offline.rcrd_cust_vhcl_info rcv
        |  left join
        |     datalake_cdm.dw_cust_cdm_accnt cdm
        |         on rcv.ucid=cdm.ucid
        |  left join (
        |     select
        |      onliemobile
        |      ,ciamid
        |      ,row_number()over(PARTITION BY onliemobile order by ciamid desc) as rn
        |     from datalake_cdm.dw_cust_cdm_digital_accnt
        | ) cdm_digital
        | on cdm.mobilephone=cdm_digital.onliemobile and cdm_digital.rn=1
        | left join
        |     temp_test_drive_dealer_view last_test_drive
        |        on last_test_drive.ucid=rcv.ucid and last_test_drive.rn=1
        | left join
        |     temp_showroom_visit_dealer_view last_showroom_visit
        |        on last_showroom_visit.ucid=rcv.ucid and last_showroom_visit.rn=1
        |) cvi
        |        on tag_exploded.unify_id=cvi.fin) src left join temp_cust_cdm_view cust on
        |        src.ucid=cust.ucid
        |""".stripMargin).write.mode(SaveMode.Overwrite).insertInto("revr_bmbs_dmp_offline.dmp_offline_dcff_tags_with_info")

/*  spark.sql(
      """
        |select *,
        |date_format(date_add(current_date,-1),'yyyyMMdd') as dt
        |from revr_bmbs_dmp_offline.dmp_offline_dcff_tags_with_info
        |""".stripMargin).write.mode(SaveMode.Overwrite).insertInto("revr_bmbs_dmp_offline.dmp_offline_dcff_tags_with_info_partition")
  send_slack_info_base_on_query_with_spark("select dt,count(1) from revr_bmbs_dmp_offline.dmp_offline_dcff_tags_with_info_partition where dt=date_format(date_add(current_date,-1),'yyyyMMdd') group by dt", "revr_bmbs_dmp_offline.dmp_offline_dcff_tags_with_info_partition record in dt(%s) count:%s")
*/
}

In [0]:
run_execute_func_in_notebook(execute_func,"mmc_store/tag_explode")